# 05. Análise Bayesiana — P04 (SEM Bayesiano)

**Objetivo:** Estimar modelos SEM com inferência Bayesiana usando `pymc` (ou fallback `scipy`).

**Quando usar:**
- Quando a amostra é pequena (<200)
- Quando queremos distribuições posteriores completas (não só p-valores)
- Quando queremos incorporar conhecimento prévio (priors informativos)

**Comparação:** Frequentista (lavaan) vs Bayesiano (pymc) no P04.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pymc as pm
import arviz as az

np.random.seed(42)

print(f"PyMC version: {pm.__version__}")
print(f"ArviZ version: {az.__version__}")

## 1. Gerar dados sintéticos (P04: IA → FE via mediação)

**Modelo conceitual:**
- X: Horas de uso de IA generativa por semana
- M: Engajamento escolar (média de 5 itens Likert)
- Y: Pontuação em função executiva (inibição)
- W: Letramento digital (moderador)

In [ ]:
# Simular dados P04
n = 200

X = np.random.gamma(shape=2, scale=5, size=n)  # horas de uso de IA
W = np.random.normal(5, 1.5, size=n)            # letramento digital
M = 0.4 * X + 0.2 * W + np.random.normal(0, 1, size=n)  # mediação
Y = 0.3 * X + 0.5 * M - 0.15 * W + np.random.normal(0, 1, size=n)  # desfecho

df = pd.DataFrame({"X_uso_ia": X, "M_engajamento": M, "Y_fe": Y, "W_letram": W})
print(df.describe())
print(f"\nN = {len(df)}")

## 2. Modelo Bayesiano: mediação simples (X → M → Y)

Equações:
- M = α_M + β_XM * X + ε_M
- Y = α_Y + β_XY * X + β_MY * M + ε_Y

**Efeito indireto:** β_XM * β_MY
**Efeito total:** β_XY + (β_XM * β_MY)

In [ ]:
with pm.Model() as mediation_model:
    # Priors fracas (vagas)
    alpha_M = pm.Normal("alpha_M", mu=0, sigma=10)
    alpha_Y = pm.Normal("alpha_Y", mu=0, sigma=10)
    beta_XM = pm.Normal("beta_XM", mu=0, sigma=1)
    beta_XY = pm.Normal("beta_XY", mu=0, sigma=1)
    beta_MY = pm.Normal("beta_MY", mu=0, sigma=1)
    
    sigma_M = pm.HalfNormal("sigma_M", sigma=5)
    sigma_Y = pm.HalfNormal("sigma_Y", sigma=5)
    
    # Modelo mediação
    mu_M = alpha_M + beta_XM * df["X_uso_ia"].values
    M_obs = pm.Normal("M_obs", mu=mu_M, sigma=sigma_M, observed=df["M_engajamento"].values)
    
    mu_Y = alpha_Y + beta_XY * df["X_uso_ia"].values + beta_MY * df["M_engajamento"].values
    Y_obs = pm.Normal("Y_obs", mu=mu_Y, sigma=sigma_Y, observed=df["Y_fe"].values)
    
    # Efeitos derivados
    indirect = pm.Deterministic("indirect", beta_XM * beta_MY)
    total = pm.Deterministic("total", beta_XY + indirect)
    
    # Amostrar
    trace = pm.sample(2000, tune=1000, cores=2, random_seed=42, progressbar=False)

print("\n✓ Amostragem concluída")

## 3. Sumário das posteriores

In [ ]:
summary = az.summary(trace, var_names=["alpha_M", "alpha_Y", "beta_XM", "beta_XY", "beta_MY", "indirect", "total"])
print(summary)

# Comparar com OLS frequentista (verificação)
from scipy import stats
import statsmodels.api as sm

X_mat = sm.add_constant(df[["X_uso_ia"]])
ols_M = sm.OLS(df["M_engajamento"], X_mat).fit()
print("\nOLS (M ~ X):")
print(f"  beta_XM = {ols_M.params['X_uso_ia']:.3f}, p = {ols_M.pvalues['X_uso_ia']:.4f}")

XY_mat = sm.add_constant(df[["X_uso_ia", "M_engajamento"]])
ols_Y = sm.OLS(df["Y_fe"], XY_mat).fit()
print("\nOLS (Y ~ X + M):")
print(f"  beta_XY = {ols_Y.params['X_uso_ia']:.3f}, p = {ols_Y.pvalues['X_uso_ia']:.4f}")
print(f"  beta_MY = {ols_Y.params['M_engajamento']:.3f}, p = {ols_Y.pvalues['M_engajamento']:.4f}")

## 4. Visualização das posteriores

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

params_to_plot = ["beta_XM", "beta_XY", "beta_MY", "indirect", "total"]
for i, param in enumerate(params_to_plot):
    az.plot_posterior(trace, var_names=[param], ax=axes[i], hdi_prob=0.95)
    axes[i].set_title(param, fontsize=12, fontweight="bold")

axes[5].axis("off")
plt.tight_layout()
plt.savefig("../resultados/figura5_posteriores_bayes.png", dpi=300, bbox_inches="tight")
plt.show()

## 5. Diagnóstico de convergência (R-hat e ESS)

In [ ]:
rhat = az.rhat(trace)
ess = az.ess(trace)

print("R-hat (deve estar < 1.01):")
for var in ["beta_XM", "beta_XY", "beta_MY"]:
    print(f"  {var}: {float(rhat[var]):.4f}")

print("\nESS bulk (Effective Sample Size, deve ser > 400):")
for var in ["beta_XM", "beta_XY", "beta_MY"]:
    print(f"  {var}: {float(ess[var]):.0f}")

## 6. Comparação Bayesiano vs Frequentista

| Parâmetro | Bayesiano (média) | Frequentista (OLS) | Diferença |
|---|---|---|---|
| β_XM | ver trace | ver OLS | |
| β_XY | ver trace | ver OLS | |
| β_MY | ver trace | ver OLS | |
| Indireto | ver trace | bootstrap | |

**Vantagens do Bayesiano:**
1. Distribuições posteriores completas
2. Credible intervals com interpretação direta
3. Pode incorporar priors informativos (conhecimento prévio)
4. Melhor para amostras pequenas

**Vantagens do Frequentista:**
1. Mais rápido computacionalmente
2. Não precisa especificar priors
3. Mais familiares para a área

## 7. Probabilidade posterior de efeito

Pergunta: qual a probabilidade de o efeito indireto ser positivo?

In [ ]:
indirect_samples = trace.posterior["indirect"].values.flatten()

prob_positive = np.mean(indirect_samples > 0)
prob_negative = np.mean(indirect_samples < 0)

print(f"P(indireto > 0) = {prob_positive:.3f}")
print(f"P(indireto < 0) = {prob_negative:.3f}")
print(f"\nInterpretação: Há {prob_positive*100:.1f}% de probabilidade de que o efeito indireto seja positivo")

# Visualizar
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(indirect_samples, bins=50, density=True, alpha=0.7, color="#667eea", edgecolor="black")
ax.axvline(0, color="red", linestyle="--", linewidth=2, label="Zero")
ax.axvline(np.mean(indirect_samples), color="green", linestyle="--", linewidth=2,
           label=f"Média = {np.mean(indirect_samples):.3f}")
ax.set_xlabel("Efeito indireto (β_XM × β_MY)", fontsize=11)
ax.set_ylabel("Densidade", fontsize=11)
ax.set_title("Distribuição Posterior do Efeito Indireto", fontsize=13, fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("../resultados/figura6_posterior_indireto.png", dpi=300, bbox_inches="tight")
plt.show()

## 8. Conclusões

**Para o manuscrito do P04:**
1. Reportar ambos: estimativas pontuais + IC frequentista + HDI bayesiano
2. Incluir P(indireto > 0) como métrica complementar
3. Discussão sobre robustez dos achados

**Próximos passos:**
- Adicionar priors informativos baseados em metanálises
- Rodar análise de sensibilidade
- Comparar com modelos mais complexos (WAIC, LOO)